# 03 Demo: aspect-level sentiment for a hotel manager

Pipeline shown end to end: pasted reviews > mask personal details > split into sentences > tag aspects > aspect-aware sentiment (DeBERTa ABSA) > per-aspect summary with the source sentences.
**Colab:** run all cells, then open the link printed by the last cell (use a GPU runtime if available). After editing `src/`, Runtime > Restart session.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/hotel-review-aspect-sentiment/notebooks')
except ImportError:
    pass

In [ ]:
# !pip install -q gradio transformers sentencepiece     # uncomment if an import fails
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import torch
from src import models as M
from src.demo import analyse_reviews, summarise, evidence, split_reviews, build_app, SAMPLE_REVIEWS, mask_personal_details

OUT = Path.cwd().parent / "outputs"; OUT.mkdir(exist_ok=True)
device = 0 if torch.cuda.is_available() else -1
absa = M.HFAbsaModel(device=device)

## 1. Walk through the pipeline on the sample reviews

In [ ]:
print(mask_personal_details("Call me on +971 50 123 4567 or mail john.smith@example.com, see www.example.com"))
reviews = split_reviews(SAMPLE_REVIEWS)
table = analyse_reviews(reviews, absa)
table

In [ ]:
summary = summarise(table)
summary.to_csv(OUT / "demo_summary.csv", index=False)
table.to_csv(OUT / "demo_table.csv", index=False)
summary

In [ ]:
# Every number links back to source sentences (the report's answer to LLM hallucination risk).
for asp in summary["aspect"][:3]:
    print(asp, "->", evidence(table, asp, "negative"))

## 2. Why aspect-aware matters: one sentence, two verdicts
An aspect-blind model gives the whole sentence one label. The aspect-aware model is asked about each aspect separately.

In [ ]:
tests = ["The room was excellent, but the staff were rude.",
         "The breakfast wasn't bad but the wifi didn't work.",
         "Not the cleanest hotel but the location is perfect and staff were helpful."]
rob = M.HFSentimentModel(device=device)
rows = analyse_reviews(tests, absa)
rows["aspect-blind RoBERTa"] = rob.predict(rows)
rows = rows.rename(columns={"sentiment": "aspect-aware ABSA"})
rows.to_csv(OUT / "demo_aspect_aware_vs_blind.csv", index=False)
pd.set_option("display.max_colwidth", 80)
rows[["sentence", "aspect", "aspect-aware ABSA", "aspect-blind RoBERTa"]]

## 3. Launch the web app
Opens a Gradio page (a public link appears when `share=True`). Paste reviews separated by blank lines.

In [ ]:
app = build_app(absa)
app.launch(share=True, debug=False)